# neologism-embedding-learning: Colab runner

A thin driver. It mounts Drive, clones the repo, installs it and calls the CLI scripts. Every stage writes its caches and results to `OUT_DIR` on Drive and skips work that is already done, so after a disconnect you re-run the setup cells and then the cell that was interrupted.

**Before running:** choose Runtime → Change runtime type → GPU (an L4 or A100 is plenty for a 0.5B model). Optionally add `HF_TOKEN` under the 🔑 Secrets tab and enable notebook access. The model is public, so the token only raises download rate limits. The token stays in Secrets: the cells read it and never print it.

Runtimes have not been measured on Colab yet. Most of the cost is about 1,700 greedy generations of up to 512 tokens at batch size 1 for full Experiment 2, plus about 430 more for the LoRA baseline, so expect a few hours on an L4.

In [ ]:
REPO_URL = "https://github.com/BobbY-24/neologism-embedding-learning.git"
BRANCH = "main"
OUT_DIR = "/content/drive/MyDrive/neologism-embedding-learning/outputs"
REPO_DIR = "/content/neologism-embedding-learning"
MODEL_DIR = "/content/models/Qwen2.5-0.5B-Instruct"  # local disk; weights are not copied to Drive
MODE = "full"

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# The secret value is never printed or written to disk.
from google.colab import userdata


def secret(key):
    try:
        return userdata.get(key)
    except Exception:  # secret missing or notebook access not granted
        return None


hf = secret("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf  # read by huggingface_hub
print(f"HF_TOKEN: {'set' if hf else 'not set (fine: the model is public)'}")
del hf

In [ ]:
import subprocess


def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end="")
    r.check_returncode()


if os.path.isdir(REPO_DIR):
    git("-C", REPO_DIR, "fetch", "-q", "origin")
    git("-C", REPO_DIR, "checkout", "-q", BRANCH)
    git("-C", REPO_DIR, "pull", "-q", "--ff-only")
else:
    git("clone", "-q", "--branch", BRANCH, REPO_URL, REPO_DIR)
git("-C", REPO_DIR, "log", "-1", "--oneline")
os.chdir(REPO_DIR)

In [ ]:
# Keeps Colab's preinstalled CUDA torch; installs this package plus peft and pytest.
!pip install -q -e "{REPO_DIR}[test]"
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))"

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("Qwen/Qwen2.5-0.5B-Instruct", local_dir=MODEL_DIR)
# Every later `!python` call inherits these.
os.environ["QWEN_MODEL_PATH"] = MODEL_DIR
os.environ["NEOLOGISM_OUTPUT_ROOT"] = OUT_DIR

## Phase 0: tests and a smoke run
The CPU unit tests, then the whole pipeline in `smoke` mode: 1 training pair, 2 held-out prompts and 12-token generations, written to `OUT_DIR/smoke` so it never mixes with real results. Takes a few minutes. Its numbers are meaningless; it only checks that every stage runs on this GPU.

In [ ]:
!python -m pytest -q
SMOKE = f"NEOLOGISM_OUTPUT_ROOT={OUT_DIR}/smoke"
!{SMOKE} python scripts/run_experiment2.py --mode smoke --stage all
!{SMOKE} python scripts/run_interp.py --mode smoke
!{SMOKE} python scripts/run_lora_baseline.py --mode smoke
!{SMOKE} python scripts/make_figures.py --mode smoke

## Phase 1: full multi-seed Experiment 2
Uses the frozen 20-pair / 36-prompt split. The committed screening outputs are reused, and the run stops if the split it rebuilds differs from `results/experiment_2/prepared_full_dataset/dataset_manifest.json`. Five genuine seeds, three shuffled-label seeds, position tests on 20 prompts, and learning curves at N = 0, 1, 2, 5, 10, 20 over three seeds. All conditions, base included, are generated in this runtime so they share one device and dtype. Run the cells in order; each one resumes.

In [ ]:
!python scripts/run_experiment2.py --mode {MODE} --stage train

In [ ]:
!python scripts/run_experiment2.py --mode {MODE} --stage evaluate

In [ ]:
!python scripts/run_experiment2.py --mode {MODE} --stage positions

In [ ]:
!python scripts/run_experiment2.py --mode {MODE} --stage learning-curve

In [ ]:
!python scripts/run_experiment2.py --mode {MODE} --stage analyze
print(open(f"{OUT_DIR}/experiment_2/experiment_summary.md").read())

## Phase 2: interpretability of the learned token
Needs only the Phase 1 `train` stage. For every genuine seed it writes:
- nearest vocabulary neighbours of E_w, its initialization and the update E_w − E_w⁽⁰⁾;
- the logit lens at the token position and at the stop-decision position after the base model's concise answer;
- activation patching of each layer's output at the token position, from the trained run into the untrained run, scored by the teacher-forced log p(`<|im_end|>`).

Output goes to `OUT_DIR/experiment_2/interp/full/`. These are forward passes only, so it is quick.

In [ ]:
!python scripts/run_interp.py --mode {MODE}

## Phase 3: LoRA baseline
Uses the same pairs, DPO objective and prompts. The token stays at its untrained vector and only the adapters learn. There are two arms in `experiments/experiment_2/configs/lora_baseline.json`, each with three seeds: `lora_matched` (rank 1 on one `v_proj`, 1,024 parameters against the neologism's 896) and `lora_r8_qv` (rank 8 on q/v in every layer, about 541k parameters, reported rather than matched). Each arm is evaluated with the token template and on plain prompts, which tests whether the adapters leak onto prompts without the token. Needs Phase 1 `evaluate`.

In [ ]:
!python scripts/run_lora_baseline.py --mode {MODE}

## Phase 4: figures and a results bundle
Writes the interpretability and LoRA figures next to Experiment 2's figures in `OUT_DIR/experiment_2/figures`. It then zips the small result files (CSV, JSON, PNG and summary; no embeddings, adapters or caches) into `OUT_DIR/results_full.zip`, which you can copy into the repo's `results/experiment_2/full/`.

In [ ]:
!python scripts/make_figures.py --mode {MODE}
import glob

from IPython.display import Image, display

for path in sorted(glob.glob(f"{OUT_DIR}/experiment_2/figures/*.png")):
    print(os.path.basename(path))
    display(Image(path, width=640))

In [ ]:
import zipfile

root = f"{OUT_DIR}/experiment_2"
keep = (".csv", ".json", ".png", ".md")
with zipfile.ZipFile(f"{OUT_DIR}/results_full.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder, _, files in os.walk(root):
        for name in files:
            path = os.path.join(folder, name)
            relative = os.path.relpath(path, root)
            if relative.startswith("smoke") or "generation_cache" in name or "candidate_outputs" in name:
                continue
            if name.endswith(keep) and "/quick/" not in relative and "/smoke/" not in relative:
                bundle.write(path, relative)
print(f"{OUT_DIR}/results_full.zip")